# Best Practices — Functions

This notebook is the *library* of the three-notebook solution. It defines the two transformation functions used by the pipeline and the tests:

* `location_to_region(location_col)` — maps a location string column to a region (`europe`, `asia`, `americas`, `unknown`, `other`) using native Spark expressions, not a Python UDF.
* `compute_user_region_stats(users, answers, min_answers)` — per-user answer count, average score, region; filtered to users with at least `min_answers` answers.

Nothing here creates a `SparkSession`, reads data, or triggers an action — the file is pure transformation logic. The pipeline notebook (`Best Practices - Pipeline.ipynb`) and the test notebook (`Best Practices - Tests.ipynb`) pull these definitions into their own kernels with the IPython `%run` magic:

```
%run "./Best Practices - Functions.ipynb"
```

In a real production codebase this file would be a `transformations.py` module imported with a plain `from my_pipeline.transformations import ...`. The notebook + `%run` form is the closest equivalent when the deployment artifact is a notebook (e.g. a Databricks Workflow).

In [ ]:
import pyspark.sql.functions as f
from pyspark.sql import Column, DataFrame

### `location_to_region`

Takes a `Column` (the location string) and returns a `Column` expression mapping it to a region label. Implemented with `f.when(...).when(...).otherwise(...)` over `rlike` substring checks — no row-by-row Python execution.

The calling convention `location_to_region(f.col('location'))` is identical to the original Python-UDF version, so existing callers do not need to change.

In [ ]:
def location_to_region(location_col: Column) -> Column:
    location_lower = f.lower(location_col)
    return (
        f.when(
            location_lower.rlike(r'france|germany|norway|sweden|czech|poland|spain|italy|netherlands'),
            f.lit('europe'),
        )
        .when(
            location_lower.rlike(r'china|japan|india|singapore|thailand'),
            f.lit('asia'),
        )
        .when(
            location_lower.rlike(r'usa|united states|brazil|canada|mexico'),
            f.lit('americas'),
        )
        .when(location_col.isNull(), f.lit('unknown'))
        .otherwise(f.lit('other'))
    )

### `compute_user_region_stats`

Per-user aggregation over the joined users + answers data, with the region attached. Returns a DataFrame restricted to users with at least `min_answers` answers.

* `users: DataFrame` — must contain `user_id` and `location` columns
* `answers: DataFrame` — must contain `user_id` and `score` columns
* `min_answers: int = 5` — threshold on the per-user answer count

In [ ]:
def compute_user_region_stats(
    users: DataFrame,
    answers: DataFrame,
    min_answers: int = 5,
) -> DataFrame:
    return (
        users
        .join(answers, 'user_id')
        .withColumn('region', location_to_region(f.col('location')))
        .groupBy('user_id', 'region')
        .agg(
            f.count('*').alias('answer_count'),
            f.avg('score').alias('avg_score'),
        )
        .filter(f.col('answer_count') >= min_answers)
    )